In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("Churn.csv")

df['Geography'] = LabelEncoder().fit_transform(df['Geography'])
df['Gender'] = LabelEncoder().fit_transform(df['Gender'])

df_model = df.drop(columns=['RowNumber', 'CustomerId', 'Surname', 'Card Type'])

realistic_features = ['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 
                       'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 
                       'EstimatedSalary']

y = df_model['Exited']
X_realistic = df_model[realistic_features]

X_train_r, X_test_r, y_train, y_test = train_test_split(
    X_realistic, y, test_size=0.2, random_state=42, stratify=y)

rf_realistic = RandomForestClassifier(n_estimators=200, class_weight='balanced', 
                                        random_state=42)
rf_realistic.fit(X_train_r, y_train)

proba_r = rf_realistic.predict_proba(X_test_r)[:, 1]
pred_r = rf_realistic.predict(X_test_r)

print("Setup complete")

Setup complete


Assign risk tiers based on predicted probability

In [3]:
import pandas as pd

results = X_test_r.copy()
results['Actual_Exited'] = y_test.values
results['Predicted_Probability'] = proba_r

# Simple, explainable tiers — thresholds chosen to be round and easy to communicate
def risk_tier(p):
    if p >= 0.7:
        return 'High Risk'
    elif p >= 0.4:
        return 'Medium Risk'
    else:
        return 'Low Risk'

results['Risk_Tier'] = results['Predicted_Probability'].apply(risk_tier)

tier_summary = results.groupby('Risk_Tier').agg(
    Customers=('Actual_Exited', 'count'),
    Actual_Churn_Rate=('Actual_Exited', 'mean')
).round(3)

print(tier_summary)

             Customers  Actual_Churn_Rate
Risk_Tier                                
High Risk          213              0.803
Low Risk          1440              0.078
Medium Risk        347              0.357


Illustrative cost-benefit calculation (assumptions clearly stated)

In [4]:
# ASSUMPTIONS (illustrative only — no real financial data exists for this dataset)
retention_offer_cost = 50       # cost of a retention offer/discount per customer contacted
avg_customer_value = 1000       # illustrative annual value of retaining a customer
retention_success_rate = 0.30   # assumed % of contacted at-risk customers who are successfully retained

high_risk = results[results['Risk_Tier'] == 'High Risk']
n_high_risk = len(high_risk)
actual_churners_in_high_risk = high_risk['Actual_Exited'].sum()

cost_of_campaign = n_high_risk * retention_offer_cost
customers_saved = actual_churners_in_high_risk * retention_success_rate
value_saved = customers_saved * avg_customer_value

print(f"High-risk customers flagged: {n_high_risk}")
print(f"Actual churners within that group: {actual_churners_in_high_risk}")
print(f"Campaign cost (contacting all high-risk): ${cost_of_campaign:,.0f}")
print(f"Estimated customers saved: {customers_saved:.0f}")
print(f"Estimated value saved: ${value_saved:,.0f}")
print(f"Net benefit: ${value_saved - cost_of_campaign:,.0f}")

High-risk customers flagged: 213
Actual churners within that group: 171
Campaign cost (contacting all high-risk): $10,650
Estimated customers saved: 51
Estimated value saved: $51,300
Net benefit: $40,650
